# Cleaning Notebook

This notebook shows the data cleaning workflow used to transform raw SMS exports into a consistent dataset for analysis and modeling.

The notebook covers:
- loading raw SMS export data
- tagging failed, balance, airtime, and adjustment messages
- filtering out non-behavioral OTP and promotional text
- removing duplicates and invalid records
- saving the cleaned dataset and a data quality report

**Instructions:** Run the cells in order. Each cell includes comments explaining what it does and what output to expect.

In [ ]:
# Cell 1: Import required libraries
# This cell loads the libraries needed for data processing and text matching.
# Run this first to set up the environment.

from pathlib import Path
import pandas as pd
import re

print("Libraries imported successfully.")

In [ ]:
# Cell 2: Set up file paths
# This cell defines the paths to input and output files relative to the notebook location.
# Expected output: Confirmation of paths.

base_dir = Path.cwd().parent
raw_path = base_dir / '1_Data_Collection' / 'raw_data.csv'
cleaned_path = base_dir / '2_Data_Cleaning' / 'cleaned_data.csv'
quality_path = base_dir / '2_Data_Cleaning' / 'data_quality_report.csv'

print(f"Raw data path: {raw_path}")
print(f"Cleaned data will be saved to: {cleaned_path}")
print(f"Quality report will be saved to: {quality_path}")

In [ ]:
# Cell 3: Load raw data
# This cell reads the raw SMS export data and ensures text columns are properly formatted.
# Expected output: Number of rows and columns loaded.

raw = pd.read_csv(raw_path)
print(f'Loaded raw data with {len(raw):,} rows and {len(raw.columns)} columns.')

# Ensure text columns are treated consistently.
raw['content'] = raw['content'].astype(str)

In [ ]:
# Cell 3.5: Anonymize sensitive information
# This cell removes or replaces personal information like names, phone numbers, account numbers, and emails.
# Expected output: Preview of anonymized content.

def anonymize_content(text: str) -> str:
    """Anonymize personal information in SMS content."""
    # Patterns for different types of sensitive information
    patterns = [
        (r'\b\d{9,12}\b', '[PHONE_NUMBER]'),  # Phone numbers (9-12 digits)
        (r'\b\d{10,16}\b', '[ACCOUNT_NUMBER]'),  # Account numbers (10-16 digits)
        (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]'),  # Email addresses
        (r'\b[A-Z][a-z]+\s+[A-Z][a-z]+\b', '[FULL_NAME]'),  # Full names (First Last)
        (r'\b[A-Z][a-z]+\b(?=\s+(?:received|sent|transferred|paid))', '[FIRST_NAME]'),  # First names before transaction verbs
    ]
    
    anonymized = str(text)
    for pattern, replacement in patterns:
        anonymized = re.sub(pattern, replacement, anonymized, flags=re.IGNORECASE)
    
    return anonymized

raw['content'] = raw['content'].apply(anonymize_content)

print("Sensitive information anonymized in content.")
print("Anonymization patterns applied:")
print("- Phone numbers replaced with [PHONE_NUMBER]")
print("- Account numbers replaced with [ACCOUNT_NUMBER]")
print("- Email addresses replaced with [EMAIL]")
print("- Names replaced with [FULL_NAME] or [FIRST_NAME]")

display(raw[['content']].head())

In [ ]:
# Cell 4: Define regex patterns for message categorization
# This cell sets up regular expressions to identify different types of SMS messages.
# Expected output: List of patterns defined.

failed_re = re.compile(r'failed|rejected|echec|echoue|annule|canceled|cancelled', re.IGNORECASE)
airtime_re = re.compile(r'\bairtime\b', re.IGNORECASE)
balance_re = re.compile(r'\b(solde|balance)\b', re.IGNORECASE)
adjustment_re = re.compile(r'\badjustment|ajustement\b', re.IGNORECASE)
otp_re = re.compile(r'\b(code|otp|login|verifier|verification)\b', re.IGNORECASE)
promo_re = re.compile(r'\b(bonus|promo|promotion|cadeau|win|gagner|scholarship)\b', re.IGNORECASE)

print("Regex patterns defined for:")
print("- Failed transactions")
print("- Airtime purchases")
print("- Balance inquiries")
print("- Account adjustments")
print("- OTP/security codes")
print("- Promotional messages")

In [ ]:
# Cell 5: Tag transaction status and types
# This cell applies the regex patterns to categorize messages and standardize transaction types.
# Expected output: Preview of tagged data.

raw['status'] = raw['content'].apply(lambda text: 'failed' if failed_re.search(str(text)) else 'success')

raw['transaction_type'] = raw['transaction_type'].astype(str).str.lower()
raw.loc[raw['content'].str.contains(airtime_re, na=False), 'transaction_type'] = 'airtime'
raw.loc[raw['content'].str.contains(balance_re, na=False), 'transaction_type'] = 'balance'
raw.loc[raw['content'].str.contains(adjustment_re, na=False), 'transaction_type'] = 'adjustment'

print("Status and transaction types tagged.")
print(f"Status distribution: {raw['status'].value_counts().to_dict()}")
print(f"Transaction type distribution: {raw['transaction_type'].value_counts().head().to_dict()}")

display(raw[['content', 'status', 'transaction_type']].head())

In [ ]:
# Cell 6: Filter out non-behavioral messages
# This cell removes OTP and promotional messages that don't represent user activity.
# Expected output: Number of messages removed.

otp_mask = raw['content'].str.contains(otp_re, na=False)
promo_mask = raw['content'].str.contains(promo_re, na=False)
cleaned = raw[~otp_mask & ~promo_mask].copy()

print(f'Removed {otp_mask.sum() + promo_mask.sum():,} OTP/promotional rows.')
print(f'Remaining rows: {len(cleaned):,}')

display(cleaned.head())

In [ ]:
# Cell 7: Remove duplicates and invalid records
# This cell drops exact duplicates and rows with missing critical fields.
# Expected output: Number of rows after cleaning.

cleaned = cleaned.dropna(subset=['datetime', 'content'])
cleaned = cleaned.drop_duplicates(subset=['user_id', 'datetime', 'content', 'amount', 'transaction_type'])

print(f'After removing duplicates and missing data: {len(cleaned):,} rows')

In [ ]:
# Cell 8: Filter to relevant transaction types
# This cell keeps only transaction categories that represent user activity.
# Expected output: Final transaction type distribution.

keep_types = {'receive', 'withdraw', 'transfer', 'payment', 'airtime', 'balance', 'adjustment'}
cleaned = cleaned[cleaned['transaction_type'].isin(keep_types)].copy()

print(f'Kept transaction types: {sorted(keep_types)}')
print(f'Final transaction type distribution:\n{cleaned["transaction_type"].value_counts()}')

In [ ]:
# Cell 9: Handle amount fields
# This cell fills missing amounts for balance messages and ensures numeric conversion.
# Expected output: Summary of amount processing.

cleaned.loc[(cleaned['transaction_type'] == 'balance') & (cleaned['amount'].isna()), 'amount'] = 0
cleaned['amount'] = pd.to_numeric(cleaned['amount'], errors='coerce')
cleaned = cleaned[cleaned['amount'].fillna(0) >= 0]
cleaned = cleaned[~cleaned['amount'].isna() | (cleaned['transaction_type'] == 'balance')]

print(f'Final dataset: {len(cleaned):,} rows')
print(f'Amount range: {cleaned["amount"].min():.2f} to {cleaned["amount"].max():.2f}')

In [ ]:
# Cell 10: Save cleaned data and quality report
# This cell saves the final cleaned dataset and generates a missing value report.
# Expected output: Confirmation of saved files and quality summary.

cleaned.to_csv(cleaned_path, index=False)
quality = cleaned.isna().mean().reset_index()
quality.columns = ['column', 'missing_rate']
quality.to_csv(quality_path, index=False)

print(f'Cleaned dataset saved to: {cleaned_path}')
print(f'Quality report saved to: {quality_path}')
print(f'Cleaned dataset contains {len(cleaned):,} rows.')

display(cleaned.head())
display(quality.head())